In [ ]:
# CIR benchmark suite: standalone GPU/JAX Kaggle notebook.
# The single cell contains JAX kernels and plotting/orchestration code directly.
# It does not clone, write, or import the local repository.

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import time
from datetime import datetime, timezone
from functools import partial
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("JAX_ENABLE_X64", "true")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
from scipy.stats import ncx2

jax.config.update("jax_enable_x64", True)

RUN_MODE = os.environ.get("CIR_SUITE_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("CIR_SUITE_RUN_MODE must be 'full' or 'smoke'")

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "cir_benchmark_suite_jax_outputs"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
BACKUP_DIR = OUT_DIR / "backups"
LATEST_PARTIAL_BACKUP = BACKUP_DIR / "latest_partial.zip"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 120339106
FINAL_TIME = 1.0
SHARED = dict(kappa=2.0, theta=0.02, x0=0.02)
REGIMES = {
    "A": {"sigma": 0.10},
    "B": {"sigma": 0.20},
    "C": {"sigma": 0.282842712474619},
    "D": {"sigma": 0.50},
    "E": {"sigma": 0.80},
}
STRONG_SCHEMES = ["FTE", "ProjEuler", "KL", "KLM"]

def parse_int_list(value: str) -> list[int]:
    return [int(part.strip()) for part in value.split(",") if part.strip()]


def reference_grid_from_env(default_power: int) -> tuple[int, int]:
    if "CIR_SUITE_REFERENCE_N_STEPS" in os.environ and "CIR_SUITE_REFERENCE_POWER" not in os.environ:
        n_steps = int(os.environ["CIR_SUITE_REFERENCE_N_STEPS"])
        if n_steps <= 0 or n_steps & (n_steps - 1):
            raise ValueError("CIR_SUITE_REFERENCE_N_STEPS must be a positive power of two")
        return int(np.log2(n_steps)), n_steps
    reference_power = int(os.environ.get("CIR_SUITE_REFERENCE_POWER", str(default_power)))
    if reference_power <= 0:
        raise ValueError("CIR_SUITE_REFERENCE_POWER must be positive")
    return reference_power, 2**reference_power


def fig3_config(default_paths: int, default_reference_power: int, default_n_a: int, default_levels: list[int]) -> dict:
    levels_text = ",".join(str(level) for level in default_levels)
    return dict(
        n_paths=int(os.environ.get("CIR_SUITE_FIG3_N_PATHS", str(default_paths))),
        reference_power=int(os.environ.get("CIR_SUITE_FIG3_REFERENCE_POWER", str(default_reference_power))),
        n_paper_a_values=int(os.environ.get("CIR_SUITE_FIG3_N_A_VALUES", str(default_n_a))),
        levels=parse_int_list(os.environ.get("CIR_SUITE_FIG3_LEVELS", levels_text)),
    )


if RUN_MODE == "smoke":
    N_PATHS = int(os.environ.get("CIR_SUITE_N_PATHS", "256"))
    REFERENCE_POWER, REFERENCE_N_STEPS = reference_grid_from_env(default_power=9)
    COARSE_N_STEPS = parse_int_list(os.environ.get("CIR_SUITE_COARSE_N_STEPS", "8,16,32"))
    REGIME_LIST = ["A", "B", "C", "D", "E"]
    PATH_BATCH_SIZE = int(os.environ.get("CIR_SUITE_PATH_BATCH_SIZE", str(min(N_PATHS, 64))))
    CHUNK_STEPS = int(os.environ.get("CIR_SUITE_CHUNK_STEPS", "128"))
    FIG3_CONFIG = fig3_config(64, 12, 5, [4, 5])
else:
    N_PATHS = int(os.environ.get("CIR_SUITE_N_PATHS", "2048"))
    REFERENCE_POWER, REFERENCE_N_STEPS = reference_grid_from_env(default_power=22)
    COARSE_N_STEPS = parse_int_list(os.environ.get("CIR_SUITE_COARSE_N_STEPS", "8,16,32,64,128,256"))
    REGIME_LIST = ["A", "B", "C", "D", "E"]
    PATH_BATCH_SIZE = int(os.environ.get("CIR_SUITE_PATH_BATCH_SIZE", "512"))
    CHUNK_STEPS = int(os.environ.get("CIR_SUITE_CHUNK_STEPS", str(2**14)))
    FIG3_CONFIG = fig3_config(1000, 16, 40, [4, 5, 6, 7, 8, 9])

if any(REFERENCE_N_STEPS % n_steps != 0 for n_steps in COARSE_N_STEPS):
    raise ValueError("Every coarse step count must divide the HH reference step count")
if REFERENCE_N_STEPS % CHUNK_STEPS != 0:
    raise ValueError("CIR_SUITE_CHUNK_STEPS must divide the HH reference step count")
if PATH_BATCH_SIZE <= 0 or CHUNK_STEPS <= 0:
    raise ValueError("CIR_SUITE_PATH_BATCH_SIZE and CIR_SUITE_CHUNK_STEPS must be positive")

RESUME = os.environ.get("CIR_SUITE_RESUME", "1").strip() not in {"0", "false", "False", "no", "NO"}
CHECKPOINT_EVERY_BATCHES = int(os.environ.get("CIR_SUITE_CHECKPOINT_EVERY_BATCHES", "1"))
BACKUP_EVERY_BLOCKS = int(os.environ.get("CIR_SUITE_BACKUP_EVERY_BLOCKS", "1"))
BATCH_CSV = RES_DIR / "jax_strong_error_batch_contributions.csv"
PARTIAL_STRONG_CSV = RES_DIR / "jax_strong_error_partial.csv"

RUN_CONFIG = {
    "run_mode": RUN_MODE,
    "n_paths": N_PATHS,
    "reference": "HH",
    "reference_power": REFERENCE_POWER,
    "reference_n_steps": REFERENCE_N_STEPS,
    "reference_dt": FINAL_TIME / REFERENCE_N_STEPS,
    "coarse_n_steps": COARSE_N_STEPS,
    "regimes": REGIME_LIST,
    "path_batch_size": PATH_BATCH_SIZE,
    "chunk_steps": CHUNK_STEPS,
    "schemes": STRONG_SCHEMES,
    "resume": RESUME,
    "checkpoint_every_batches": CHECKPOINT_EVERY_BATCHES,
    "backup_every_blocks": BACKUP_EVERY_BLOCKS,
    "batch_contribution_csv": str(BATCH_CSV),
    "partial_diagnostics_csv": str(PARTIAL_STRONG_CSV),
    "fig3_config": FIG3_CONFIG,
}
RUN_CONFIG_HASH_PAYLOAD = json.dumps(
    {
        key: value
        for key, value in RUN_CONFIG.items()
        if key not in {"resume", "checkpoint_every_batches", "backup_every_blocks", "run_config_hash"}
    },
    sort_keys=True,
    separators=(",", ":"),
)
RUN_CONFIG_HASH = hashlib.sha256(RUN_CONFIG_HASH_PAYLOAD.encode("utf-8")).hexdigest()[:16]
RUN_CONFIG["run_config_hash"] = RUN_CONFIG_HASH



def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    tmp_path.replace(path)


def append_csv_rows(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def read_csv_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def run_hash_payload(config: dict) -> dict:
    ignored = {"resume", "checkpoint_every_batches", "backup_every_blocks", "run_config_hash"}
    return {key: value for key, value in config.items() if key not in ignored}


def make_run_config_hash(config: dict) -> str:
    payload = json.dumps(run_hash_payload(config), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def batch_row_uid(row: dict) -> tuple:
    return (
        row.get("run_config_hash", ""),
        row.get("regime", ""),
        row.get("batch_start", ""),
        row.get("scheme", ""),
        row.get("n_steps", ""),
    )


def current_batch_rows(path: Path) -> list[dict]:
    unique = {}
    for row in read_csv_rows(path):
        if row.get("run_config_hash") == RUN_CONFIG_HASH:
            unique[batch_row_uid(row)] = row
    return list(unique.values())


def completed_batch_keys(path: Path) -> set[tuple[str, str]]:
    counts = {}
    expected = len(STRONG_SCHEMES) * len(COARSE_N_STEPS)
    for row in current_batch_rows(path):
        key = (row["regime"], row["batch_start"])
        counts[key] = counts.get(key, 0) + 1
    return {key for key, count in counts.items() if count >= expected}


def aggregate_strong_batch_rows(batch_rows: list[dict]) -> list[dict]:
    groups = {}
    for row in batch_rows:
        key = (row["regime"], row["scheme"], row["n_steps"])
        if key not in groups:
            groups[key] = {
                "rows": [],
                "sum_abs": 0.0,
                "sum_sq": 0.0,
                "paths": 0,
                "step_count_sum": 0.0,
                "backstop_count": 0.0,
                "runtime_s": 0.0,
            }
        group = groups[key]
        group["rows"].append(row)
        group["sum_abs"] += float(row["sum_abs"])
        group["sum_sq"] += float(row["sum_sq"])
        group["paths"] += int(row["batch_size"])
        group["step_count_sum"] += float(row["step_count_sum"])
        group["backstop_count"] += float(row["backstop_count"])
        group["runtime_s"] += float(row["runtime_s"])

    regime_order = {name: idx for idx, name in enumerate(REGIME_LIST)}
    scheme_order = {name: idx for idx, name in enumerate(STRONG_SCHEMES)}
    diagnostics = []
    for key in sorted(groups, key=lambda item: (regime_order[item[0]], scheme_order[item[1]], int(item[2]))):
        first = groups[key]["rows"][0]
        paths = groups[key]["paths"]
        step_count_sum = groups[key]["step_count_sum"]
        diagnostics.append({
            "run_config_hash": RUN_CONFIG_HASH,
            "run_mode": RUN_MODE,
            "regime": first["regime"],
            "scheme": first["scheme"],
            "scheme_variant": first["scheme_variant"],
            "reference": "HH",
            "reference_power": REFERENCE_POWER,
            "reference_n_steps": REFERENCE_N_STEPS,
            "reference_dt": FINAL_TIME / REFERENCE_N_STEPS,
            "n_paths": N_PATHS,
            "completed_paths": paths,
            "path_batch_size": PATH_BATCH_SIZE,
            "chunk_steps": CHUNK_STEPS,
            "n_steps": int(first["n_steps"]),
            "dt": float(first["dt"]),
            "l1": groups[key]["sum_abs"] / paths,
            "l2": float(np.sqrt(groups[key]["sum_sq"] / paths)),
            "runtime_s": groups[key]["runtime_s"],
            "timing_scope": "batch-checkpointed-regime",
            "mean_steps_per_path": step_count_sum / paths,
            "backstop_fraction": groups[key]["backstop_count"] / max(step_count_sum, 1.0),
        })
    return diagnostics


def write_progress_manifest(stage: str, batch_rows: list[dict], diagnostics: list[dict]) -> None:
    expected_batches = len(REGIME_LIST) * len(range(0, N_PATHS, PATH_BATCH_SIZE))
    completed_batches = len({
        (row["regime"], row["batch_start"])
        for row in batch_rows
    })
    manifest = {
        "stage": stage,
        "updated_utc": utc_timestamp(),
        "run_config_hash": RUN_CONFIG_HASH,
        "batch_contribution_rows": len(batch_rows),
        "diagnostic_rows": len(diagnostics),
        "completed_batches": completed_batches,
        "expected_batches": expected_batches,
        "completed_batch_fraction": completed_batches / expected_batches if expected_batches else 0.0,
        "batch_contribution_csv": str(BATCH_CSV),
        "partial_diagnostics_csv": str(PARTIAL_STRONG_CSV),
    }
    with (RES_DIR / "progress_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)


def refresh_strong_outputs(final: bool = False) -> list[dict]:
    batch_rows = current_batch_rows(BATCH_CSV)
    diagnostics = aggregate_strong_batch_rows(batch_rows)
    if diagnostics:
        write_csv(PARTIAL_STRONG_CSV, diagnostics)
        for regime_name in REGIME_LIST:
            regime_rows = [r for r in diagnostics if r["regime"] == regime_name]
            if regime_rows:
                write_csv(RES_DIR / f"jax_strong_error_regime_{regime_name}.csv", regime_rows)
        if final:
            write_csv(RES_DIR / "jax_strong_error_all_regimes.csv", diagnostics)
    write_progress_manifest("final" if final else "partial", batch_rows, diagnostics)
    return diagnostics


def fit_loglog_order(step_sizes, errors):
    h = np.asarray(step_sizes, dtype=float)
    e = np.maximum(np.asarray(errors, dtype=float), 1e-300)
    return float(np.polyfit(np.log(h), np.log(e), 1)[0])


def cir_delta(kappa, theta, sigma):
    return 4.0 * kappa * theta / sigma**2


def kl_alpha(kappa, theta, sigma):
    return (4.0 * kappa * theta - sigma**2) / 8.0


def aggregate_dW(dW_fine, factor):
    n_paths, n_fine = dW_fine.shape
    return dW_fine.reshape((n_paths, n_fine // factor, factor)).sum(axis=2)


def hh_step(x, kappa, theta, sigma, dt, dW):
    floor = 0.25 * sigma**2 * dt
    r1 = jnp.maximum(
        0.5 * sigma * jnp.sqrt(dt),
        jnp.sqrt(jnp.maximum(floor, x)) + 0.5 * sigma * dW,
    )
    x_hat = r1 * r1 + dt * (kappa * (theta - x) - 0.25 * sigma**2)
    return jnp.maximum(x_hat, 0.0)


def hh_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    def step(x, dW_col):
        return hh_step(x, kappa, theta, sigma, dt, dW_col), None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x, _ = jax.lax.scan(step, x0, dW.T)
    return x


def fte_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    def step(x_aux, dW_col):
        x_pos = jnp.maximum(x_aux, 0.0)
        x_next = x_aux + kappa * (theta - x_pos) * dt + sigma * jnp.sqrt(x_pos) * dW_col
        return x_next, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x_aux, _ = jax.lax.scan(step, x0, dW.T)
    return jnp.maximum(x_aux, 0.0)


def projected_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    y_floor = 0.5 * sigma * dt ** 0.5
    def step(y, dW_col):
        y_safe = jnp.maximum(y, y_floor)
        y_hat = y_safe + (alpha / y_safe - 0.5 * kappa * y_safe) * dt + 0.5 * sigma * dW_col
        return jnp.maximum(y_hat, y_floor), None
    y0 = jnp.full((dW.shape[0],), max(np.sqrt(X0), y_floor), dtype=jnp.float64)
    y, _ = jax.lax.scan(step, y0, dW.T)
    return y * y


def kl_uniform_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    decay = np.exp(-kappa * dt)
    def step(x, dW_col):
        return decay * (jnp.sqrt(x + 2.0 * alpha * dt) + 0.5 * sigma * dW_col) ** 2, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x, _ = jax.lax.scan(step, x0, dW.T)
    return x


def soft_zero_threshold(kappa, theta, dt_max, rho=2.0):
    return theta * (1.0 - np.exp(-kappa * dt_max)) / rho


def kl_adaptive_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, dt_max, dW_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    X_zero = soft_zero_threshold(kappa, theta, dt_max, rho)
    dW_fine = jnp.asarray(dW_fine, dtype=jnp.float64)
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    x0 = jnp.full((n_paths,), X0, dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(2, dtype=jnp.int64)

    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)

    def body(state):
        x, pos, counters = state
        active = pos < n_fine
        remaining_steps = n_fine - pos
        remaining_time = remaining_steps * dt_fine
        in_soft_zero = active & (x < X_zero)
        in_splitting = active & (~in_soft_zero)

        x_for_soft = jnp.minimum(x, X_zero * 0.999999)
        dt_sz = -jnp.log((X_zero - theta) / (x_for_soft - theta)) / kappa
        if alpha < 0.0:
            dt_adaptive = safety * x / (2.0 * abs(alpha))
            split_h = jnp.minimum(jnp.minimum(dt_adaptive, dt_max), remaining_time)
        else:
            split_h = jnp.minimum(dt_max, remaining_time)

        h_cont = jnp.where(in_soft_zero, jnp.minimum(dt_sz, remaining_time), 0.0)
        h_cont = jnp.where(in_splitting, split_h, h_cont)
        m = jnp.floor(h_cont / dt_fine).astype(jnp.int64)
        m = jnp.where(active, jnp.maximum(m, 1), 0)
        m = jnp.minimum(m, remaining_steps)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]

        decay = jnp.exp(-kappa * h)
        x_soft = decay * x + theta * (1.0 - decay)
        inside_sqrt = jnp.maximum(x + 2.0 * alpha * h, 0.0)
        x_split = decay * (jnp.sqrt(inside_sqrt) + 0.5 * sigma * dW) ** 2
        x_next = jnp.where(in_soft_zero, x_soft, jnp.where(in_splitting, x_split, x))

        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(in_soft_zero)],
            dtype=jnp.int64,
        )
        return x_next, pos + m, counters

    x, _, counters = jax.lax.while_loop(cond, body, (x0, pos0, counters0))
    total = int(counters[0])
    return x, {
        "n_steps_total": total,
        "mean_steps_per_path": total / n_paths,
        "soft_zero_fraction": int(counters[1]) / max(total, 1),
    }


def implicit_step(y, h, dW, alpha, beta, gamma):
    a = 1.0 - beta * h
    u = y + gamma * dW
    return (u + jnp.sqrt(u * u + 4.0 * a * alpha * h)) / (2.0 * a)


def projected_backstop_step(y, h, dW, alpha, beta, gamma):
    y_floor = gamma * jnp.sqrt(h)
    y_hat = y + (alpha / y + beta * y) * h + gamma * dW
    return jnp.maximum(y_hat, y_floor)


def klm_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, h_max, dW_fine, rho=64.0):
    alpha = kl_alpha(kappa, theta, sigma)
    beta = -0.5 * kappa
    gamma = 0.5 * sigma
    use_implicit = alpha > 0.0
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    h_min = h_max / rho
    m_min = max(int(round(h_min / dt_fine)), 1)
    m_max = max(int(np.floor(h_max / dt_fine)), 1)
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    y0 = jnp.full((n_paths,), jnp.sqrt(X0), dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(3, dtype=jnp.int64)
    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)
    def body(state):
        y, pos, counters = state
        active = pos < n_fine
        h_prop = h_max * jnp.minimum(1.0, jnp.abs(y))
        m = jnp.floor(h_prop / dt_fine).astype(jnp.int64)
        min_triggered = m < m_min
        m = jnp.where(min_triggered, m_min, jnp.minimum(m, m_max))
        m = jnp.minimum(m, n_fine - pos)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]
        y_explicit = jnp.where(h > 0.0, y + (alpha / y + beta * y) * h + gamma * dW, y)
        y_imp = implicit_step(y, h, dW, alpha, beta, gamma)
        y_proj = projected_backstop_step(y, h, dW, alpha, beta, gamma)
        y_backstop = jnp.where(use_implicit, y_imp, y_proj)
        neg_retake = (~min_triggered) & (y_explicit <= 0.0)
        use_backstop = min_triggered | neg_retake
        y_next = jnp.where(use_backstop, y_backstop, y_explicit)
        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(active & min_triggered), jnp.sum(active & neg_retake)],
            dtype=jnp.int64,
        )
        return y_next, pos + m, counters
    y, _, counters = jax.lax.while_loop(cond, body, (y0, pos0, counters0))
    total = int(counters[0])
    return y * y, {
        "n_steps_total": total,
        "n_backstop_min": int(counters[1]),
        "n_backstop_neg": int(counters[2]),
        "backstop_fraction": (int(counters[1]) + int(counters[2])) / max(total, 1),
    }


def choose_kl_adaptive_interval(x, remaining_steps, kappa, theta, sigma, h_values, dt_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    x_zero = theta * (1.0 - jnp.exp(-kappa * h_values)) / rho
    in_soft_zero = x < x_zero
    x_for_soft = jnp.minimum(x, x_zero * 0.999999)
    dt_soft = -jnp.log((x_zero - theta) / (x_for_soft - theta)) / kappa
    dt_adaptive = safety * x / (2.0 * jnp.abs(alpha))
    split_h = jnp.minimum(jnp.minimum(dt_adaptive, h_values), remaining_steps * dt_fine)
    h_cont = jnp.where(in_soft_zero, jnp.minimum(dt_soft, remaining_steps * dt_fine), split_h)
    m = jnp.floor(h_cont / dt_fine).astype(jnp.int64)
    m = jnp.where(remaining_steps > 0, jnp.maximum(m, 1), 0)
    m = jnp.minimum(m, remaining_steps)
    return m, jnp.where(remaining_steps > 0, in_soft_zero, False)


def choose_klm_interval(y, remaining_steps, h_values, dt_fine, rho=64.0):
    h_min = h_values / rho
    m_min = jnp.maximum(jnp.rint(h_min / dt_fine).astype(jnp.int64), 1)
    m_max = jnp.maximum(jnp.floor(h_values / dt_fine).astype(jnp.int64), 1)
    h_prop = h_values * jnp.minimum(1.0, jnp.abs(y))
    m_raw = jnp.floor(h_prop / dt_fine).astype(jnp.int64)
    min_triggered = m_raw < m_min
    m = jnp.where(min_triggered, m_min, jnp.minimum(m_raw, m_max))
    m = jnp.where(remaining_steps > 0, jnp.minimum(m, remaining_steps), 0)
    return m, jnp.where(remaining_steps > 0, min_triggered, False)


def initial_stream_state(batch_size, kappa, theta, sigma, factors, h_values, use_kl_adaptive):
    n_levels = len(COARSE_N_STEPS)
    factors = jnp.asarray(factors, dtype=jnp.int64).reshape((n_levels, 1))
    h_values = jnp.asarray(h_values, dtype=jnp.float64).reshape((n_levels, 1))
    level_shape = (n_levels, batch_size)
    reference = jnp.full((batch_size,), SHARED["x0"], dtype=jnp.float64)
    fixed_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    fixed_remaining = jnp.broadcast_to(factors, level_shape)
    fte_aux = jnp.full(level_shape, SHARED["x0"], dtype=jnp.float64)
    proj_y = jnp.maximum(jnp.sqrt(SHARED["x0"]), 0.5 * sigma * jnp.sqrt(h_values)) * jnp.ones(level_shape, dtype=jnp.float64)
    kl_x = jnp.full(level_shape, SHARED["x0"], dtype=jnp.float64)
    kl_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    if use_kl_adaptive:
        kl_m, kl_soft = choose_kl_adaptive_interval(
            kl_x, REFERENCE_N_STEPS, kappa, theta, sigma, h_values, FINAL_TIME / REFERENCE_N_STEPS
        )
    else:
        kl_m = jnp.zeros(level_shape, dtype=jnp.int64)
        kl_soft = jnp.zeros(level_shape, dtype=bool)
    kl_remaining = kl_m
    kl_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    kl_soft_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_y = jnp.full(level_shape, jnp.sqrt(SHARED["x0"]), dtype=jnp.float64)
    klm_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    klm_m, klm_min = choose_klm_interval(
        klm_y, REFERENCE_N_STEPS, h_values, FINAL_TIME / REFERENCE_N_STEPS
    )
    klm_remaining = klm_m
    klm_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_min_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_neg_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    return (
        reference,
        fixed_acc,
        fixed_remaining,
        fte_aux,
        proj_y,
        kl_x,
        kl_acc,
        kl_remaining,
        kl_m,
        kl_soft,
        kl_total,
        kl_soft_total,
        klm_y,
        klm_acc,
        klm_remaining,
        klm_m,
        klm_min,
        klm_total,
        klm_min_total,
        klm_neg_total,
    )


def make_dW_chunk(regime_index, batch_start, chunk_start, batch_size, chunk_steps, dt_fine):
    key = jax.random.PRNGKey(MASTER_SEED)
    key = jax.random.fold_in(key, int(regime_index))
    key = jax.random.fold_in(key, int(batch_start))
    key = jax.random.fold_in(key, int(chunk_start))
    return jax.random.normal(key, (batch_size, chunk_steps), dtype=jnp.float64) * jnp.sqrt(dt_fine)


@partial(jax.jit, static_argnames=("use_kl_adaptive",))
def stream_strong_chunk(
    state,
    dW_chunk,
    start_index,
    reference_n_steps,
    dt_fine,
    kappa,
    theta,
    sigma,
    factors,
    h_values,
    use_kl_adaptive,
):
    factors = factors.astype(jnp.int64)
    h_values = h_values.astype(jnp.float64)
    alpha = kl_alpha(kappa, theta, sigma)

    def one_fine_step(carry, inputs):
        dW_col, fine_index = inputs
        remaining_after = reference_n_steps - fine_index - 1
        (
            reference,
            fixed_acc,
            fixed_remaining,
            fte_aux,
            proj_y,
            kl_x,
            kl_acc,
            kl_remaining,
            kl_m,
            kl_soft,
            kl_total,
            kl_soft_total,
            klm_y,
            klm_acc,
            klm_remaining,
            klm_m,
            klm_min,
            klm_total,
            klm_min_total,
            klm_neg_total,
        ) = carry

        reference = hh_step(reference, kappa, theta, sigma, dt_fine, dW_col)

        fixed_acc_next = fixed_acc + dW_col[None, :]
        fixed_remaining_next = fixed_remaining - 1
        fixed_land = fixed_remaining_next <= 0

        fte_pos = jnp.maximum(fte_aux, 0.0)
        fte_candidate = fte_aux + kappa * (theta - fte_pos) * h_values + sigma * jnp.sqrt(fte_pos) * fixed_acc_next
        fte_aux = jnp.where(fixed_land, fte_candidate, fte_aux)

        y_floor = 0.5 * sigma * jnp.sqrt(h_values)
        proj_safe = jnp.maximum(proj_y, y_floor)
        proj_candidate = proj_safe + (alpha / proj_safe - 0.5 * kappa * proj_safe) * h_values + 0.5 * sigma * fixed_acc_next
        proj_y = jnp.where(fixed_land, jnp.maximum(proj_candidate, y_floor), proj_y)

        if use_kl_adaptive:
            kl_acc_next = kl_acc + dW_col[None, :]
            kl_remaining_next = kl_remaining - 1
            kl_land = kl_remaining_next <= 0
            kl_h = kl_m * dt_fine
            kl_decay = jnp.exp(-kappa * kl_h)
            kl_soft_candidate = kl_decay * kl_x + theta * (1.0 - kl_decay)
            kl_inside = jnp.maximum(kl_x + 2.0 * alpha * kl_h, 0.0)
            kl_split_candidate = kl_decay * (jnp.sqrt(kl_inside) + 0.5 * sigma * kl_acc_next) ** 2
            kl_candidate = jnp.where(kl_soft, kl_soft_candidate, kl_split_candidate)
            kl_x = jnp.where(kl_land, kl_candidate, kl_x)
            new_kl_m, new_kl_soft = choose_kl_adaptive_interval(
                kl_x, remaining_after, kappa, theta, sigma, h_values, dt_fine
            )
            kl_total = kl_total + jnp.sum(kl_land, axis=1)
            kl_soft_total = kl_soft_total + jnp.sum(kl_land & kl_soft, axis=1)
            kl_acc = jnp.where(kl_land, 0.0, kl_acc_next)
            kl_remaining = jnp.where(kl_land, new_kl_m, kl_remaining_next)
            kl_m = jnp.where(kl_land, new_kl_m, kl_m)
            kl_soft = jnp.where(kl_land, new_kl_soft, kl_soft)
        else:
            kl_decay = jnp.exp(-kappa * h_values)
            kl_inside = jnp.maximum(kl_x + 2.0 * alpha * h_values, 0.0)
            kl_candidate = kl_decay * (jnp.sqrt(kl_inside) + 0.5 * sigma * fixed_acc_next) ** 2
            kl_x = jnp.where(fixed_land, kl_candidate, kl_x)

        fixed_acc = jnp.where(fixed_land, 0.0, fixed_acc_next)
        fixed_remaining = jnp.where(fixed_land, factors, fixed_remaining_next)

        klm_acc_next = klm_acc + dW_col[None, :]
        klm_remaining_next = klm_remaining - 1
        klm_land = klm_remaining_next <= 0
        klm_h = klm_m * dt_fine
        beta = -0.5 * kappa
        gamma = 0.5 * sigma
        klm_explicit = jnp.where(
            klm_h > 0.0,
            klm_y + (alpha / klm_y + beta * klm_y) * klm_h + gamma * klm_acc_next,
            klm_y,
        )
        klm_implicit = implicit_step(klm_y, klm_h, klm_acc_next, alpha, beta, gamma)
        klm_projected = projected_backstop_step(klm_y, klm_h, klm_acc_next, alpha, beta, gamma)
        klm_backstop = jnp.where(alpha > 0.0, klm_implicit, klm_projected)
        klm_neg = (~klm_min) & (klm_explicit <= 0.0)
        klm_use_backstop = klm_min | klm_neg
        klm_candidate = jnp.where(klm_use_backstop, klm_backstop, klm_explicit)
        klm_y = jnp.where(klm_land, klm_candidate, klm_y)
        new_klm_m, new_klm_min = choose_klm_interval(klm_y, remaining_after, h_values, dt_fine)
        klm_total = klm_total + jnp.sum(klm_land, axis=1)
        klm_min_total = klm_min_total + jnp.sum(klm_land & klm_min, axis=1)
        klm_neg_total = klm_neg_total + jnp.sum(klm_land & klm_neg, axis=1)
        klm_acc = jnp.where(klm_land, 0.0, klm_acc_next)
        klm_remaining = jnp.where(klm_land, new_klm_m, klm_remaining_next)
        klm_m = jnp.where(klm_land, new_klm_m, klm_m)
        klm_min = jnp.where(klm_land, new_klm_min, klm_min)

        return (
            reference,
            fixed_acc,
            fixed_remaining,
            fte_aux,
            proj_y,
            kl_x,
            kl_acc,
            kl_remaining,
            kl_m,
            kl_soft,
            kl_total,
            kl_soft_total,
            klm_y,
            klm_acc,
            klm_remaining,
            klm_m,
            klm_min,
            klm_total,
            klm_min_total,
            klm_neg_total,
        ), None

    fine_indices = start_index + jnp.arange(dW_chunk.shape[1], dtype=jnp.int64)
    new_state, _ = jax.lax.scan(one_fine_step, state, (dW_chunk.T, fine_indices))
    return new_state



def plot_strong_error_figures(rows: list[dict]) -> None:
    if not rows:
        return
    for regime_name in REGIME_LIST:
        regime_rows = [r for r in rows if r["regime"] == regime_name]
        if not regime_rows:
            continue
        fig, ax = plt.subplots(figsize=(6.0, 4.4))
        for scheme in STRONG_SCHEMES:
            sr = sorted([r for r in regime_rows if r["scheme"] == scheme], key=lambda row: row["dt"])
            if not sr:
                continue
            order = fit_loglog_order([r["dt"] for r in sr], [r["l2"] for r in sr]) if len(sr) >= 2 else float("nan")
            ax.loglog([r["dt"] for r in sr], [r["l2"] for r in sr], "o-", label=f"{scheme} ({order:.2f})")
        ax.set_xlabel("step size h")
        ax.set_ylabel("strong L2 error")
        ax.set_title(f"JAX strong benchmark, regime {regime_name}, HH h_ref=2^-{REFERENCE_POWER}")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"jax_strong_error_regime_{regime_name}.pdf")
        plt.close(fig)


def run_strong_error_suite() -> list[dict]:
    dt_fine = FINAL_TIME / REFERENCE_N_STEPS
    factors_np = np.asarray([REFERENCE_N_STEPS // n_steps for n_steps in COARSE_N_STEPS], dtype=np.int64)
    h_values_np = np.asarray([FINAL_TIME / n_steps for n_steps in COARSE_N_STEPS], dtype=float)
    factors = jnp.asarray(factors_np.reshape((-1, 1)), dtype=jnp.int64)
    h_values = jnp.asarray(h_values_np.reshape((-1, 1)), dtype=jnp.float64)
    completed = completed_batch_keys(BATCH_CSV) if RESUME else set()

    for regime_index, regime_name in enumerate(REGIME_LIST):
        sigma = REGIMES[regime_name]["sigma"]
        alpha = kl_alpha(SHARED["kappa"], SHARED["theta"], sigma)
        use_kl_adaptive = alpha < 0.0
        regime_start = time.perf_counter()

        for batch_start in range(0, N_PATHS, PATH_BATCH_SIZE):
            batch_key = (regime_name, str(batch_start))
            if batch_key in completed:
                print(f"  skipping completed batch regime={regime_name}, batch_start={batch_start}")
                continue

            batch_runtime_start = time.perf_counter()
            batch_size = min(PATH_BATCH_SIZE, N_PATHS - batch_start)
            state = initial_stream_state(
                batch_size,
                SHARED["kappa"],
                SHARED["theta"],
                sigma,
                factors_np,
                h_values_np,
                use_kl_adaptive,
            )
            for chunk_start in range(0, REFERENCE_N_STEPS, CHUNK_STEPS):
                dW_chunk = make_dW_chunk(regime_index, batch_start, chunk_start, batch_size, CHUNK_STEPS, dt_fine)
                state = stream_strong_chunk(
                    state,
                    dW_chunk,
                    jnp.asarray(chunk_start, dtype=jnp.int64),
                    jnp.asarray(REFERENCE_N_STEPS, dtype=jnp.int64),
                    jnp.asarray(dt_fine, dtype=jnp.float64),
                    jnp.asarray(SHARED["kappa"], dtype=jnp.float64),
                    jnp.asarray(SHARED["theta"], dtype=jnp.float64),
                    jnp.asarray(sigma, dtype=jnp.float64),
                    factors,
                    h_values,
                    use_kl_adaptive=use_kl_adaptive,
                )
            state = jax.block_until_ready(state)
            (
                reference,
                _fixed_acc,
                _fixed_remaining,
                fte_aux,
                proj_y,
                kl_x,
                _kl_acc,
                _kl_remaining,
                _kl_m,
                _kl_soft,
                kl_total_batch,
                kl_soft_batch,
                klm_y,
                _klm_acc,
                _klm_remaining,
                _klm_m,
                _klm_min,
                klm_total_batch,
                klm_min_batch,
                klm_neg_batch,
            ) = state
            reference_np = np.asarray(reference)
            terminals = {
                "FTE": np.maximum(np.asarray(fte_aux), 0.0),
                "ProjEuler": np.asarray(proj_y) ** 2,
                "KL": np.asarray(kl_x),
                "KLM": np.asarray(klm_y) ** 2,
            }
            batch_runtime = time.perf_counter() - batch_runtime_start
            timestamp = utc_timestamp()
            batch_rows = []
            for scheme in STRONG_SCHEMES:
                diff = terminals[scheme] - reference_np[None, :]
                sum_abs = np.sum(np.abs(diff), axis=1)
                sum_sq = np.sum(diff * diff, axis=1)
                for idx, n_steps in enumerate(COARSE_N_STEPS):
                    if scheme == "KL" and use_kl_adaptive:
                        step_count_sum = float(np.asarray(kl_total_batch)[idx])
                        backstop_count = float(np.asarray(kl_soft_batch)[idx])
                        scheme_variant = "adaptive-soft-zero"
                    elif scheme == "KLM":
                        step_count_sum = float(np.asarray(klm_total_batch)[idx])
                        backstop_count = float((np.asarray(klm_min_batch) + np.asarray(klm_neg_batch))[idx])
                        scheme_variant = "adaptive-backstop"
                    else:
                        step_count_sum = float(n_steps * batch_size)
                        backstop_count = 0.0
                        scheme_variant = "uniform" if scheme == "KL" else "fixed"
                    batch_rows.append({
                        "run_config_hash": RUN_CONFIG_HASH,
                        "timestamp_utc": timestamp,
                        "run_mode": RUN_MODE,
                        "regime": regime_name,
                        "scheme": scheme,
                        "scheme_variant": scheme_variant,
                        "reference": "HH",
                        "reference_power": REFERENCE_POWER,
                        "reference_n_steps": REFERENCE_N_STEPS,
                        "reference_dt": dt_fine,
                        "n_paths": N_PATHS,
                        "path_batch_size": PATH_BATCH_SIZE,
                        "chunk_steps": CHUNK_STEPS,
                        "n_steps": n_steps,
                        "dt": FINAL_TIME / n_steps,
                        "batch_start": batch_start,
                        "batch_size": batch_size,
                        "sum_abs": float(sum_abs[idx]),
                        "sum_sq": float(sum_sq[idx]),
                        "runtime_s": batch_runtime,
                        "timing_scope": "single-path-batch",
                        "step_count_sum": step_count_sum,
                        "backstop_count": backstop_count,
                    })
            append_csv_rows(BATCH_CSV, batch_rows)
            print(
                f"  checkpointed regime={regime_name}, batch_start={batch_start}, "
                f"paths={batch_size}, runtime={batch_runtime:.1f}s"
            )
            refresh_strong_outputs(final=False)

        rows = refresh_strong_outputs(final=False)
        plot_strong_error_figures(rows)
        print(f"completed regime={regime_name} in {time.perf_counter() - regime_start:.1f}s")
        if BACKUP_EVERY_BLOCKS > 0 and (regime_index + 1) % BACKUP_EVERY_BLOCKS == 0:
            backup_outputs("latest_partial")
    return refresh_strong_outputs(final=True)


def sigma_from_a(a, kappa, lam):
    return np.sqrt(2.0 * kappa * lam * np.asarray(a, dtype=float))


def run_fig3_order_sweep() -> list[dict]:
    rows = []
    lambda_value = 0.05
    a_values = np.concatenate(([0.0], np.linspace(0.04, 1.6, FIG3_CONFIG["n_paper_a_values"])))
    for kappa in [2.0, 0.2]:
        reference_power = FIG3_CONFIG["reference_power"]
        n_fine = 2**reference_power
        dt_fine = FINAL_TIME / n_fine
        dW_fine = jax.random.normal(
            jax.random.PRNGKey(MASTER_SEED),
            (FIG3_CONFIG["n_paths"], n_fine),
            dtype=jnp.float64,
        ) * jnp.sqrt(dt_fine)
        for a in a_values:
            sigma = float(sigma_from_a(a, kappa, lambda_value))
            reference = hh_terminal_from_dW_jax(0.02**2, kappa, lambda_value, sigma, dt_fine, dW_fine)
            reference.block_until_ready()
            for level in FIG3_CONFIG["levels"]:
                hmax = 2.0 ** (-level)
                start = time.perf_counter()
                terminal, stats = klm_terminal_from_fine_dW_jax(0.02**2, kappa, lambda_value, sigma, FINAL_TIME, hmax, dW_fine)
                terminal.block_until_ready()
                runtime = time.perf_counter() - start
                diff = terminal - reference
                rows.append({
                    "kappa": kappa,
                    "a": float(a),
                    "sigma": sigma,
                    "level": level,
                    "hmax": hmax,
                    "reference_power": reference_power,
                    "n_paths": FIG3_CONFIG["n_paths"],
                    "rmse": float(jnp.sqrt(jnp.mean(diff * diff))),
                    "backstop_fraction": float(stats["backstop_fraction"]),
                    "mean_steps_per_path": float(stats["n_steps_total"] / FIG3_CONFIG["n_paths"]),
                    "runtime_s": runtime,
                })
    write_csv(RES_DIR / "jax_fig3_order_sweep.csv", rows)
    orders = []
    for kappa, a in sorted({(r["kappa"], r["a"]) for r in rows}):
        sr = [r for r in rows if r["kappa"] == kappa and r["a"] == a]
        if len(sr) >= 2:
            orders.append({"kappa": kappa, "a": a, "fitted_order": fit_loglog_order([r["hmax"] for r in sr], [r["rmse"] for r in sr])})
    write_csv(RES_DIR / "jax_fig3_fitted_orders.csv", orders)
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    for kappa in [2.0, 0.2]:
        sr = [r for r in orders if r["kappa"] == kappa and r["a"] > 0.0]
        ax.plot([r["a"] for r in sr], [r["fitted_order"] for r in sr], "o-", label=f"kappa={kappa:g}")
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    ax.axhline(1.0, color="k", ls=":", lw=0.8)
    ax.set_xlabel("a = sigma^2 / (2 kappa lambda)")
    ax.set_ylabel("fitted strong L2 order")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "jax_fig3_fitted_orders.pdf")
    plt.close(fig)
    return rows



def write_run_config() -> None:
    with (OUT_DIR / "run_config.json").open("w", encoding="utf-8") as file:
        json.dump(RUN_CONFIG, file, indent=2)


def backup_outputs(label: str = "latest_partial") -> None:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    tmp_base = WORKING_ROOT / f"{OUT_DIR.name}_{label}_backup_tmp"
    archive = shutil.make_archive(str(tmp_base), "zip", OUT_DIR)
    target = LATEST_PARTIAL_BACKUP if label == "latest_partial" else BACKUP_DIR / f"{label}.zip"
    if target.exists():
        target.unlink()
    shutil.move(archive, target)
    print(f"Backup written to {target}")


def archive_outputs() -> None:
    archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print(f"Archived outputs to {archive}")


def main() -> None:
    print(f"Standalone GPU/JAX CIR benchmark suite, RUN_MODE={RUN_MODE}")
    print("JAX devices:", jax.devices())
    print(f"Output directory: {OUT_DIR}")
    print(
        "Strong benchmark: "
        f"N_PATHS={N_PATHS}, h_ref=2^-{REFERENCE_POWER}, "
        f"REFERENCE_N_STEPS={REFERENCE_N_STEPS}, "
        f"PATH_BATCH_SIZE={PATH_BATCH_SIZE}, CHUNK_STEPS={CHUNK_STEPS}"
    )
    print(f"resume={RESUME}, run_config_hash={RUN_CONFIG_HASH}")
    write_run_config()
    if not RESUME:
        for path in [
            BATCH_CSV,
            PARTIAL_STRONG_CSV,
            RES_DIR / "jax_strong_error_all_regimes.csv",
        ]:
            if path.exists():
                path.unlink()
        for regime_name in REGIME_LIST:
            path = RES_DIR / f"jax_strong_error_regime_{regime_name}.csv"
            if path.exists():
                path.unlink()

    run_strong_error_suite()
    rows = refresh_strong_outputs(final=True)
    plot_strong_error_figures(rows)
    backup_outputs("latest_partial")
    run_fig3_order_sweep()
    archive_outputs()
    print("Done.")


main()
